## 1. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


## 2. Configuration


In [ ]:
from pathlib import Path

ASC_MODEL_PATH = "/content/drive/MyDrive/asc_1_office_products/asc_teacher_phase1"

INPUT_CHUNKS_DIR = Path("/content/drive/MyDrive/ate_1_Ofice/parts/high")
OUTPUT_ROOT = Path("/content/drive/MyDrive/asc_1_electronics_2")

CATEGORY_NAME = "office_products"
RUN_NAME = "asc1_office_products"

FIRST_CHUNK_NUMBER = 1
LAST_CHUNK_NUMBER = 10
MAX_CHUNKS_TO_RUN = None

POLAR_THRESHOLD = 0.90
NEUTRAL_THRESHOLD = 0.55

ASC_BATCH_SIZE = 512
MAX_LENGTH = 192
USE_LENGTH_BUCKETING = True

RESUME = True
FORCE_REPROCESS = False
COMPRESSION = "snappy"

PARTS_DIR = OUTPUT_ROOT / "parts"
HIGH_DIR = PARTS_DIR / "high"
LOW_DIR = PARTS_DIR / "low"
REPORT_DIR = OUTPUT_ROOT / "reports" / "asc_1"
FINAL_DIR = OUTPUT_ROOT / "final"

FINAL_HIGH_PATH = FINAL_DIR / f"high_confidence_samples_{CATEGORY_NAME}.parquet"
FINAL_LOW_PATH = FINAL_DIR / f"low_confidence_samples_{CATEGORY_NAME}.parquet"

# Tạo sẵn toàn bộ folder output để tránh lỗi:
for d in [HIGH_DIR, LOW_DIR, REPORT_DIR, FINAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)


## 3. Imports and device setup

In [3]:
import os
import re
import gc
import time
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from transformers import AutoTokenizer, AutoModelForSequenceClassification

torch.set_grad_enabled(False)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_FP16 = DEVICE == "cuda"

if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True
    try:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

print("device:", DEVICE)
if DEVICE == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))

device: cuda
gpu: Tesla T4


## 4. Load ASC teacher

In [4]:
asc_tokenizer = AutoTokenizer.from_pretrained(ASC_MODEL_PATH, use_fast=True)
asc_model = AutoModelForSequenceClassification.from_pretrained(ASC_MODEL_PATH).to(DEVICE)
asc_model.eval()

if USE_FP16:
    asc_model.half()

print("model:", ASC_MODEL_PATH)
print("labels:", asc_model.config.id2label)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model: /content/drive/MyDrive/asc_1_office_products/asc_teacher_phase1
labels: {0: 'negative', 1: 'neutral', 2: 'positive'}


## 5. Text and input helpers

ASC teacher dùng **aspect-marker format**:

```text
The battery life is excellent but the [ASP] screen [/ASP] is dim.
```


In [5]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def mark_aspect(sentence, aspect):
    sentence = clean_text(sentence)
    aspect = clean_text(aspect)

    pattern = re.compile(re.escape(aspect), flags=re.IGNORECASE)
    return pattern.sub(f"[ASP] {aspect} [/ASP]", sentence, count=1)


def parse_list_cell(value):
    if isinstance(value, list):
        return value

    if isinstance(value, np.ndarray):
        return value.tolist()

    if pd.isna(value):
        return []

    text = str(value).strip()

    if text == "" or text.lower() in {"nan", "none", "null"}:
        return []

    try:
        import ast
        parsed = ast.literal_eval(text)
        return parsed if isinstance(parsed, list) else []
    except Exception:
        return []


def normalize_aspects(value):
    aspects = parse_list_cell(value)
    return [clean_text(x) for x in aspects if clean_text(x)]

## 6. ASC inference

In [6]:
def length_sorted_indices(items):
    if not USE_LENGTH_BUCKETING:
        return np.arange(len(items))
    lengths = np.asarray([len(str(x)) for x in items])
    return np.argsort(lengths)


def predict_asc_pairs(sentences, aspects, batch_size=512, max_length=192):
    n = len(sentences)
    if n == 0:
        return []

    inputs = [
        mark_aspect(sentence, aspect)
        for sentence, aspect in zip(sentences, aspects)
    ]

    probs_all = np.zeros((n, 3), dtype=np.float32)
    order = length_sorted_indices(inputs)

    for pos in tqdm(range(0, n, batch_size), desc="asc", unit="pair", leave=False):
        idx = order[pos:pos + batch_size]
        batch = [inputs[i] for i in idx]

        enc = asc_tokenizer(
            batch,
            truncation=True,
            max_length=max_length,
            padding=True,
            return_tensors="pt",
        )
        enc = {k: v.to(DEVICE, non_blocking=True) for k, v in enc.items()}

        with torch.inference_mode():
            logits = asc_model(**enc).logits
            probs = torch.softmax(logits.float(), dim=-1).detach().cpu().numpy()

        probs_all[idx] = probs.astype(np.float32)

        del enc, logits, probs
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    results = []

    for probs in probs_all:
        confidences = probs.tolist()
        pred_id = int(np.argmax(confidences))
        confidence = float(confidences[pred_id])
        threshold = NEUTRAL_THRESHOLD if pred_id == 1 else POLAR_THRESHOLD
        is_high_confidence = confidence >= threshold
        results.append((confidences, is_high_confidence))

    return results

## 7. Chunk utilities

In [7]:
BASE_COLUMNS = [
    "parent_asin",
    "sentence_id",
    "sentence_text",
    "rating",
    "category_name",
    "gate_confidence",
    "aspects",
]


def natural_key(path):
    text = str(Path(path).name)
    return [int(x) if x.isdigit() else x.lower() for x in re.split(r"(\d+)", text)]


def discover_input_chunk_files():
    if INPUT_CHUNKS_DIR.is_file() and INPUT_CHUNKS_DIR.suffix.lower() == ".parquet":
        return [INPUT_CHUNKS_DIR]

    files = sorted(INPUT_CHUNKS_DIR.glob("*.parquet"), key=natural_key)

    if not files:
        raise FileNotFoundError(f"no parquet chunks in {INPUT_CHUNKS_DIR}")

    return files


def select_chunk_files(files):
    start = int(FIRST_CHUNK_NUMBER) - 1
    end = len(files) if LAST_CHUNK_NUMBER is None else int(LAST_CHUNK_NUMBER)

    selected = list(enumerate(files, start=1))[start:end]

    if MAX_CHUNKS_TO_RUN is not None:
        selected = selected[:int(MAX_CHUNKS_TO_RUN)]

    return selected


def chunk_id_from_path(file_path, chunk_number):
    stem = Path(file_path).stem
    stem = re.sub(r"[^A-Za-z0-9._-]+", "_", stem).strip("_")
    return stem or f"chunk_{chunk_number}"


def paths_for_chunk(chunk_id):
    high_path = HIGH_DIR / f"high_confidence_samples_{CATEGORY_NAME}_{chunk_id}.parquet"
    low_path = LOW_DIR / f"low_confidence_samples_{CATEGORY_NAME}_{chunk_id}.parquet"
    report_path = REPORT_DIR / f"asc_1_report_{CATEGORY_NAME}_{chunk_id}.txt"

    # Bảo hiểm: tạo folder cha trước khi save từng chunk
    high_path.parent.mkdir(parents=True, exist_ok=True)
    low_path.parent.mkdir(parents=True, exist_ok=True)
    report_path.parent.mkdir(parents=True, exist_ok=True)

    return high_path, low_path, report_path


def outputs_done(high_path, low_path, report_path):
    return high_path.exists() and low_path.exists() and report_path.exists()


def read_input_chunk(path):
    return pd.read_parquet(path)

## 8. Process one chunk

In [8]:
def prepare_input_df(raw_df):
    df = raw_df.copy()

    for col in ["parent_asin", "sentence_id", "sentence_text", "rating"]:
        if col not in df.columns:
            raise ValueError(f"missing required column: {col}")

    if "category_name" not in df.columns:
        df["category_name"] = CATEGORY_NAME

    if "gate_confidence" not in df.columns:
        df["gate_confidence"] = np.nan

    if "aspects" not in df.columns:
        raise ValueError("missing required column: aspects")

    df["parent_asin"] = df["parent_asin"].astype(str)
    df["sentence_id"] = df["sentence_id"].astype(str)
    df["sentence_text"] = df["sentence_text"].apply(clean_text)
    df["category_name"] = df["category_name"].fillna(CATEGORY_NAME).astype(str)
    df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
    df["gate_confidence"] = pd.to_numeric(df["gate_confidence"], errors="coerce")
    df["aspects"] = df["aspects"].apply(normalize_aspects)

    return df[BASE_COLUMNS].copy()


def build_pair_index(df):
    pair_sentences = []
    pair_aspects = []
    pair_row_ids = []

    for row_id, row in df.iterrows():
        for aspect in row["aspects"]:
            pair_sentences.append(row["sentence_text"])
            pair_aspects.append(aspect)
            pair_row_ids.append(row_id)

    return pair_sentences, pair_aspects, pair_row_ids

In [9]:
def process_one_chunk(raw_df, chunk_id):
    df = prepare_input_df(raw_df)

    # Loại bỏ các dòng không có aspect trước khi chạy ASC
    df = df[df["aspects"].apply(len) > 0].reset_index(drop=True)

    pair_sentences, pair_aspects, pair_row_ids = build_pair_index(df)

    pair_results = predict_asc_pairs(
        pair_sentences,
        pair_aspects,
        batch_size=ASC_BATCH_SIZE,
        max_length=MAX_LENGTH,
    )

    sentiments_by_row = {idx: [] for idx in df.index}
    high_flags_by_row = {idx: [] for idx in df.index}

    for row_id, (confidences, is_high_confidence) in zip(pair_row_ids, pair_results):
        sentiments_by_row[row_id].append(confidences)
        high_flags_by_row[row_id].append(is_high_confidence)

    df["sentiments"] = [sentiments_by_row[idx] for idx in df.index]
    df["aspect_high_flags"] = [high_flags_by_row[idx] for idx in df.index]

    df["is_high_confidence"] = [
        len(flags) == len(aspects) and all(flags)
        for aspects, flags in zip(df["aspects"], df["aspect_high_flags"])
    ]

    high_df = df[df["is_high_confidence"]].copy()
    low_df = df[~df["is_high_confidence"]].copy()

    high_df = high_df[
        [
            "parent_asin",
            "sentence_id",
            "sentence_text",
            "rating",
            "category_name",
            "gate_confidence",
            "aspects",
            "sentiments",
        ]
    ].reset_index(drop=True)

    low_df = low_df[
        [
            "parent_asin",
            "sentence_id",
            "sentence_text",
            "rating",
            "category_name",
            "aspects",
        ]
    ].reset_index(drop=True)

    stats = make_stats(df, high_df, low_df)

    return high_df, low_df, stats

## 9. Reports

In [10]:
def predicted_label_from_confidences(confidences):
    if not confidences:
        return None
    return int(np.argmax(confidences))


def make_stats(df, high_df, low_df):
    high_sentiments = []
    for values in high_df["sentiments"].tolist():
        high_sentiments.extend(values)

    pred_labels = [predicted_label_from_confidences(x) for x in high_sentiments]
    pred_labels = [x for x in pred_labels if x is not None]

    counts = {
        "negative": int(sum(x == 0 for x in pred_labels)),
        "neutral": int(sum(x == 1 for x in pred_labels)),
        "positive": int(sum(x == 2 for x in pred_labels)),
    }

    total_rows = len(df)
    total_high = len(high_df)
    total_low = len(low_df)
    total_aspects = int(sum(len(x) for x in df["aspects"]))
    total_high_aspects = int(len(high_sentiments))

    stats = {
        "total_rows": total_rows,
        "high_rows": total_high,
        "low_rows": total_low,
        "high_row_rate": total_high / total_rows if total_rows else 0.0,
        "total_aspects": total_aspects,
        "high_aspects": total_high_aspects,
        "high_aspect_rate": total_high_aspects / total_aspects if total_aspects else 0.0,
        "negative": counts["negative"],
        "neutral": counts["neutral"],
        "positive": counts["positive"],
    }

    denom = max(total_high_aspects, 1)
    stats["negative_rate"] = counts["negative"] / denom
    stats["neutral_rate"] = counts["neutral"] / denom
    stats["positive_rate"] = counts["positive"] / denom

    return stats


def write_report(report_path, chunk_id, stats):
    lines = [
        f"ASC report: {RUN_NAME}",
        f"category_name: {CATEGORY_NAME}",
        f"chunk_id: {chunk_id}",
        "",
        f"polar_threshold: {POLAR_THRESHOLD}",
        f"neutral_threshold: {NEUTRAL_THRESHOLD}",
        "",
        f"total_rows: {stats['total_rows']:,}",
        f"high_rows: {stats['high_rows']:,}",
        f"low_rows: {stats['low_rows']:,}",
        f"high_row_rate: {stats['high_row_rate']:.4f}",
        "",
        f"total_aspects: {stats['total_aspects']:,}",
        f"high_aspects: {stats['high_aspects']:,}",
        f"high_aspect_rate: {stats['high_aspect_rate']:.4f}",
        "",
        f"negative: {stats['negative']:,} ({stats['negative_rate']:.4f})",
        f"neutral: {stats['neutral']:,} ({stats['neutral_rate']:.4f})",
        f"positive: {stats['positive']:,} ({stats['positive_rate']:.4f})",
    ]

    Path(report_path).parent.mkdir(parents=True, exist_ok=True)
    Path(report_path).write_text("\n".join(lines), encoding="utf-8")

## 10. Run selected chunks

In [11]:
all_files = discover_input_chunk_files()
selected_files = select_chunk_files(all_files)

print(f"all chunks: {len(all_files):,}")
print(f"selected chunks: {len(selected_files):,}")

if selected_files:
    print("first selected:", selected_files[0][0], selected_files[0][1].name)
    print("last selected:", selected_files[-1][0], selected_files[-1][1].name)

all chunks: 4
selected chunks: 4
first selected: 1 chunk_1.parquet
last selected: 4 chunk_4.parquet


In [12]:
counts = {
    "done": 0,
    "skipped": 0,
    "error": 0,
}

for chunk_number, file_path in tqdm(selected_files, desc="chunks", unit="chunk"):
    chunk_id = chunk_id_from_path(file_path, chunk_number)
    high_path, low_path, report_path = paths_for_chunk(chunk_id)

    if RESUME and not FORCE_REPROCESS and outputs_done(high_path, low_path, report_path):
        counts["skipped"] += 1
        print(f"skip {chunk_id}")
        continue

    t0 = time.perf_counter()

    try:
        print(f"processing {chunk_id}: {file_path.name}")

        raw_df = read_input_chunk(file_path)
        high_df, low_df, stats = process_one_chunk(raw_df, chunk_id)

        # Bảo hiểm lần cuối trước khi ghi file
        high_path.parent.mkdir(parents=True, exist_ok=True)
        low_path.parent.mkdir(parents=True, exist_ok=True)
        report_path.parent.mkdir(parents=True, exist_ok=True)

        high_df.to_parquet(high_path, index=False, compression=COMPRESSION)
        low_df.to_parquet(low_path, index=False, compression=COMPRESSION)
        write_report(report_path, chunk_id, stats)

        counts["done"] += 1

        print(
            f"done {chunk_id}: "
            f"rows={stats['total_rows']:,}, "
            f"high={stats['high_rows']:,}, "
            f"low={stats['low_rows']:,}, "
            f"time={time.perf_counter() - t0:.1f}s"
        )

        # Dọn RAM sau khi đã lưu xong chunk
        del raw_df, high_df, low_df

    except Exception as e:
        counts["error"] += 1
        print(f"error {chunk_id}: {repr(e)}")

    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

print(counts)

chunks:   0%|          | 0/4 [00:00<?, ?chunk/s]

processing chunk_1: chunk_1.parquet


asc:   0%|          | 0/4081 [00:00<?, ?pair/s]

done chunk_1: rows=1,623,829, high=1,351,837, low=271,992, time=1425.3s
processing chunk_2: chunk_2.parquet


asc:   0%|          | 0/4076 [00:00<?, ?pair/s]

done chunk_2: rows=1,621,354, high=1,349,530, low=271,824, time=1415.6s
processing chunk_3: chunk_3.parquet


asc:   0%|          | 0/4073 [00:00<?, ?pair/s]

done chunk_3: rows=1,620,948, high=1,349,479, low=271,469, time=1396.1s
processing chunk_4: chunk_4.parquet


asc:   0%|          | 0/4080 [00:00<?, ?pair/s]

done chunk_4: rows=1,624,081, high=1,350,755, low=273,326, time=1405.8s
{'done': 4, 'skipped': 0, 'error': 0}


## 11. merge chunk outputs


In [13]:
# Đảm bảo folder vẫn tồn tại trước khi merge
for d in [HIGH_DIR, LOW_DIR, REPORT_DIR, FINAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

import pyarrow as pa
import pyarrow.parquet as pq
import gc
from pathlib import Path
from tqdm.auto import tqdm
import re


def part_files(directory):
    return sorted(Path(directory).glob("*.parquet"), key=natural_key)


def merge_parquet_files_streaming(files, out_path, compression="snappy"):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if not files:
        print("no files to merge:", out_path)
        return

    if out_path.exists():
        out_path.unlink()

    writer = None
    total_rows = 0

    try:
        for path in tqdm(files, desc=f"merge {out_path.name}", unit="file"):
            table = pq.read_table(path)

            if writer is None:
                writer = pq.ParquetWriter(
                    out_path,
                    table.schema,
                    compression=compression,
                )

            writer.write_table(table)
            total_rows += table.num_rows

            del table
            gc.collect()

    finally:
        if writer is not None:
            writer.close()

    print("saved:", out_path)
    print("rows:", total_rows)


high_files = part_files(HIGH_DIR)
low_files = part_files(LOW_DIR)

print("high part files:", len(high_files))
print("low part files:", len(low_files))


high part files: 4
low part files: 4


In [21]:
import ast
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from tqdm.auto import tqdm


FINAL_REPORT_PATH = REPORT_DIR / f"asc_1_report_{CATEGORY_NAME}_final.txt"


def parquet_row_count(path):
    return pq.ParquetFile(path).metadata.num_rows


def count_rows_from_files(files):
    return sum(parquet_row_count(path) for path in files)


def parse_list_value(value):
    if value is None:
        return []

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, list):
        return value

    if isinstance(value, str):
        text = value.strip()
        if text == "" or text.lower() in {"nan", "none", "null"}:
            return []
        return ast.literal_eval(text)

    return []


def count_aspects_from_input_files(files, batch_size=8192):
    total_aspect_rows = 0
    total_no_aspect_rows = 0
    total_aspects = 0

    for path in tqdm(files, desc="scan input aspects", unit="file"):
        pf = pq.ParquetFile(path)

        for batch in pf.iter_batches(columns=["aspects"], batch_size=batch_size):
            aspects_col = batch.column("aspects").to_pylist()

            for value in aspects_col:
                aspects = parse_list_value(value)
                n = len(aspects)

                if n > 0:
                    total_aspect_rows += 1
                    total_aspects += n
                else:
                    total_no_aspect_rows += 1

    return {
        "total_aspect_rows": total_aspect_rows,
        "total_no_aspect_rows": total_no_aspect_rows,
        "total_aspects": total_aspects,
    }


def count_sentiment_distribution_from_high_files(files, batch_size=8192):
    label_counts = {
        "negative": 0,
        "neutral": 0,
        "positive": 0,
    }

    total_high_aspects = 0

    for path in tqdm(files, desc="scan high sentiments", unit="file"):
        pf = pq.ParquetFile(path)

        if "sentiments" not in pf.schema_arrow.names:
            continue

        for batch in pf.iter_batches(columns=["sentiments"], batch_size=batch_size):
            sentiments_col = batch.column("sentiments").to_pylist()

            for row_sentiments in sentiments_col:
                row_sentiments = parse_list_value(row_sentiments)

                for confs in row_sentiments:
                    confs = parse_list_value(confs)
                    if len(confs) == 0:
                        continue

                    pred_id = int(np.argmax(confs))
                    total_high_aspects += 1

                    if pred_id == 0:
                        label_counts["negative"] += 1
                    elif pred_id == 1:
                        label_counts["neutral"] += 1
                    elif pred_id == 2:
                        label_counts["positive"] += 1

    return label_counts, total_high_aspects


input_files = discover_input_chunk_files()
high_files = part_files(HIGH_DIR)
low_files = part_files(LOW_DIR)

total_input_rows = count_rows_from_files(input_files)
high_rows = count_rows_from_files(high_files)
low_rows = count_rows_from_files(low_files)

input_aspect_stats = count_aspects_from_input_files(input_files)
label_counts, high_aspects = count_sentiment_distribution_from_high_files(high_files)

dropped_no_aspect_rows = input_aspect_stats["total_no_aspect_rows"]

high_row_rate = high_rows / total_input_rows if total_input_rows else 0.0
low_row_rate = low_rows / total_input_rows if total_input_rows else 0.0
dropped_no_aspect_rate = dropped_no_aspect_rows / total_input_rows if total_input_rows else 0.0

total_high_label_count = max(high_aspects, 1)

negative_rate = label_counts["negative"] / total_high_label_count
neutral_rate = label_counts["neutral"] / total_high_label_count
positive_rate = label_counts["positive"] / total_high_label_count

report_lines = [
    f"ASC final report: {RUN_NAME}",
    f"category_name: {CATEGORY_NAME}",
    "",
    f"asc_model_path: {ASC_MODEL_PATH}",
    f"input_chunks_dir: {INPUT_CHUNKS_DIR}",
    f"output_root: {OUTPUT_ROOT}",
    "",
    f"polar_threshold: {POLAR_THRESHOLD}",
    f"neutral_threshold: {NEUTRAL_THRESHOLD}",
    "",
    "1. Row-level statistics",
    f"total_input_rows_after_ATE: {total_input_rows:,}",
    f"rows_with_aspect: {input_aspect_stats['total_aspect_rows']:,}",
    f"rows_without_aspect_dropped: {dropped_no_aspect_rows:,}",
    f"high_confidence_rows: {high_rows:,}",
    f"low_confidence_rows: {low_rows:,}",
    "",
    "2. Confidence rates based on original ATE dataset",
    f"high_confidence_rate: {high_row_rate:.6f}",
    f"low_confidence_rate: {low_row_rate:.6f}",
    f"dropped_no_aspect_rate: {dropped_no_aspect_rate:.6f}",
    "",
    "3. Aspect-level statistics",
    f"total_aspects_in_input: {input_aspect_stats['total_aspects']:,}",
    f"high_confidence_aspects: {high_aspects:,}",
    f"high_confidence_aspect_rate: {high_aspects / input_aspect_stats['total_aspects'] if input_aspect_stats['total_aspects'] else 0.0:.6f}",
    "",
    "4. Sentiment distribution in high-confidence aspects",
    f"negative: {label_counts['negative']:,} ({negative_rate:.6f})",
    f"neutral: {label_counts['neutral']:,} ({neutral_rate:.6f})",
    f"positive: {label_counts['positive']:,} ({positive_rate:.6f})",
    "",
]

FINAL_REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
FINAL_REPORT_PATH.write_text("\n".join(report_lines), encoding="utf-8")

print("\n".join(report_lines))

scan input aspects:   0%|          | 0/4 [00:00<?, ?file/s]

scan high sentiments:   0%|          | 0/4 [00:00<?, ?file/s]

ASC final report: asc1_office_products
category_name: office_products

asc_model_path: /content/drive/MyDrive/asc_1_office_products/asc_teacher_phase1
input_chunks_dir: /content/drive/MyDrive/ate_1_Ofice/parts/high
output_root: /content/drive/MyDrive/asc_1_electronics_2

polar_threshold: 0.9
neutral_threshold: 0.55

1. Row-level statistics
total_input_rows_after_ATE: 10,343,383
rows_with_aspect: 6,490,212
rows_without_aspect_dropped: 3,853,171
high_confidence_rows: 5,401,601
low_confidence_rows: 1,088,611

2. Confidence rates based on original ATE dataset
high_confidence_rate: 0.522228
low_confidence_rate: 0.105247
dropped_no_aspect_rate: 0.372525

3. Aspect-level statistics
total_aspects_in_input: 8,350,150
high_confidence_aspects: 6,783,166
high_confidence_aspect_rate: 0.812341

4. Sentiment distribution in high-confidence aspects
negative: 1,925,578 (0.283876)
neutral: 6,961 (0.001026)
positive: 4,850,627 (0.715098)

